# Task 3 Handling missing data

## Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import cross_val_predict
from xgboost import XGBRegressor
from catboost import CatBoostRegressor, Pool

## Auxiliary functions

In [2]:
def cMSE(y, y_hat, c):
    err = y-y_hat
    err = (1-c)*err**2 + c*np.maximum(0,err)**2
    return np.sum(err)/err.shape[0]
    
def plot_y_yhat(y_test, y_pred, plot_title="plot"):
    label = 'SurvivalTime'
    MAX = 500

    if len(y_test) > MAX:
        idx = np.random.choice(len(y_test), MAX, replace=False)
    else:
        idx = np.arange(len(y_test))

    plt.figure(figsize=(6, 6))
    x0 = np.min(y_test[idx])
    x1 = np.max(y_test[idx])
    plt.scatter(y_test[idx], y_pred[idx], alpha=0.7)
    plt.xlabel(f'True {label}')
    plt.ylabel(f'Predicted {label}')
    plt.plot([x0, x1], [x0, x1], color='red')
    plt.title(f'True vs Predicted {label}')
    plt.axis('square')
    plt.savefig(plot_title + '.pdf')
    plt.show()
    
def save_predictions(predictions, path):
    df = pd.DataFrame({
        'id': range(len(predictions)),
        '0': predictions
    })
    
    df.to_csv(path, index=False)
    
def model_metrics(y_true, y_pred):
    error = y_true - y_pred
    metrics = {
        'Max Error': np.max(np.abs(error)),
        'Min Error': np.min(np.abs(error)),
        'Mean Error': np.mean(np.abs(error)),
        'Std of Error': np.std(error)
    }
    return metrics
    
def evaluate_model(X, y, c, model, show_cmse=True, show_plot=False):
    X_train, X_test, y_train, y_test, c_train, c_test = train_test_split(X, y, c, test_size=0.2, random_state=30)
    
    y_pred_cv = cross_val_predict(model, X_train, y_train, cv=10)
    cMSE_cv = cMSE(y_train, y_pred_cv, c_train)
    
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    cMSE_test = cMSE(y_test, y_pred, c_test)
    
    if show_cmse:
        print(f"CV cMSE: {cMSE_cv}")
        print(f"Test cMSE: {cMSE_test}")
    
    if show_plot:
        plot_y_yhat(y_test, y_pred, plot_title="y_yhat_plot")
    
    return model, cMSE_cv, cMSE_test, model_metrics(y_test, y_pred)

## Data preprocessing

In [3]:
df = pd.read_csv('./mlNOVA/train_data.csv')
df.drop(columns=['id'], inplace=True)
df = df[df['SurvivalTime'].notna()]

In [4]:
X = df.drop(columns=['SurvivalTime', 'Censored'])
y = df['SurvivalTime']
c = df['Censored']

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"c shape: {c.shape}")

X = np.array(X)
y = np.array(y)
c = np.array(c)

X shape: (240, 7)
y shape: (240,)
c shape: (240,)


## Task 3.1 Missing data imputation

### Completing missing data

#### Function for evaluating imputation

In [5]:
def evaluate_imputation(X, y, c, model, method='simple', strategy='mean', n_neighbors=5, show_cmse=True, show_metrics=False, show_plot=False):
    if method == 'simple':
        imputer = SimpleImputer(strategy=strategy)
    elif method == 'knn':
        imputer = KNNImputer(n_neighbors=n_neighbors)
    elif method == 'iterative':
        imputer = IterativeImputer()
        
    X_imp = imputer.fit_transform(X)
    
    # Train-test split
    X_train_imp, X_test_imp, y_train, y_test, c_train, c_test = train_test_split(X_imp, y, c, test_size=0.2, random_state=30)
    
    # Cross validation
    y_pred_cv = cross_val_predict(model, X_train_imp, y_train, cv=10)
    cMSE_cv = cMSE(y_train, y_pred_cv, c_train)
    
    model.fit(X_train_imp, y_train)
    
    y_pred_imp = model.predict(X_test_imp)
    cMSE_test = cMSE(y_test, y_pred_imp, c_test)
    
    if show_cmse:
        print(f"CV cMSE: {cMSE_cv}")
        print(f"Test cMSE: {cMSE_test}")
    
    # Evaluate performance
    if show_metrics:
        metrics = evaluate_model(y_test, y_pred_imp)
        print(metrics.T.to_string(index=False, header=False))
    
    if show_plot:
        plot_y_yhat(y_test, y_pred_imp, plot_title="y_yhat_plot")
    
    return model, cMSE_cv

### Evaluating imputation on the baseline

Pipeline for Linear Regression with Standard Scaler

In [177]:
linear_reg = make_pipeline(StandardScaler(), LinearRegression())

#### Simple imputer

##### Mean

In [178]:
imp_simple_mean_model, imp_simple_mean_cmse = evaluate_imputation(X, y, c, model=linear_reg, method='simple', strategy='mean', show_metrics=True, show_plot=True)

##### Median

In [179]:
imp_simple_median_model, imp_simple_median_cmse = evaluate_imputation(X, y, c, model=linear_reg, method='simple', strategy='median', show_plot=False)

##### Most frequent

In [180]:
imp_simple_most_frequent_model, imp_simple_most_frequent_cmse = evaluate_imputation(X, y, c, model=linear_reg, method='simple', strategy='most_frequent', show_plot=False)

##### Mean strategy on best model from task 2 (KNN with k=29)

K = 29

In [181]:
knn_reg = make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=29))
imp_simple_mean_model_knn, imp_simple_mean_cmse_knn = evaluate_imputation(X, y, c, model=knn_reg, method='simple', strategy='mean')

Determine best K for KNN regressor

In [182]:
k_values = [9, 11, 13, 15, 17, 19, 21, 23, 25, 27, 29, 31, 33, 35, 37]
knn_scores = []

for k in k_values:
    knn_reg = make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=k))
    imp_knn_model, imp_knn_cmse = evaluate_imputation(X, y, c, model=knn_reg, method='simple', strategy='mean', show_cmse=False, show_plot=False)
    knn_scores.append(imp_knn_cmse)

plt.plot(k_values, knn_scores, marker='o')
plt.xlabel("K")
plt.ylabel("cMSE")
plt.title("KNN Regression: K vs. cMSE")
plt.show()

In Simple imputation by mean, best K for KNN regressor is 9

In [183]:
knn_reg = make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=9))
imp_simple_mean_model_knn, imp_simple_mean_cmse_knn = evaluate_imputation(X, y, c, model=knn_reg, method='simple', strategy='mean')

#### KNN imputer

##### Find best K for KNN imputer with cross validation

In [184]:
k_values = [3, 5, 7, 9, 11, 13, 15, 17, 19, 21, 23, 25, 27, 29, 31]
knn_scores = []

for k in k_values:
    imp_knn_model, imp_knn_cmse = evaluate_imputation(X, y, c, model=linear_reg, method='knn', n_neighbors=k, show_cmse=False, show_plot=False)
    knn_scores.append(imp_knn_cmse)

plt.plot(k_values, knn_scores, marker='o')
plt.xlabel("K")
plt.ylabel("cMSE")
plt.title("KNN imputation: K vs. cMSE")
plt.show()

Best K for KNN imputer is 19

In [185]:
imp_knn_model_knn, imp_knn_cmse_knn = evaluate_imputation(X, y, c, model=linear_reg, method='knn', n_neighbors=19)

##### KNN imputation with k=19 on best model from task 2 (KNN with k=29)

K = 29

In [186]:
knn_reg = make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=29))
imp_knn_model_knn, imp_knn_cmse_knn = evaluate_imputation(X, y, c, model=knn_reg, method='knn', n_neighbors=19)

Determine the best K for the KNN regressor

In [193]:
k_values = [7, 9, 11, 13, 15, 17, 19, 21, 22, 23, 25, 27, 29, 31, 33]
knn_scores = []

for k in k_values:
    knn_reg = make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=k))
    imp_knn_model, imp_knn_cmse = evaluate_imputation(X, y, c, model=knn_reg, method='knn', n_neighbors=19, show_cmse=False, show_plot=False)
    knn_scores.append(imp_knn_cmse)

plt.plot(k_values, knn_scores, marker='o')
plt.xlabel("K")
plt.ylabel("cMSE")
plt.title("KNN Regression: K vs. cMSE")
plt.show()

In KNN imputation, best K for KNN regressor is 11

In [196]:
knn_reg = make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=11))
imp_knn_model_knn, imp_knn_cmse_knn = evaluate_imputation(X, y, c, model=knn_reg, method='knn', n_neighbors=19)

#### Iterative imputer

##### Evaluation

In [197]:
imp_iterative_model, imp_iterative_cmse = evaluate_imputation(X, y, c, model=linear_reg, method='iterative')

##### Iterative imputation on best model from task 2 (KNN with k = 29)

K = 29

In [198]:
knn_reg = make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=29))
imp_iterative_model_knn, imp_iterative_cmse_knn = evaluate_imputation(X, y, c, model=knn_reg, method='iterative')

Determine best K for KNN regressor

In [199]:
k_values = [9, 11, 13, 15, 17, 19, 21, 23, 25, 27, 29, 31, 33, 35, 37, 39]
knn_scores = []

for k in k_values:
    knn_reg = make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=k))
    imp_knn_model, imp_knn_cmse = evaluate_imputation(X, y, c, model=knn_reg, method='iterative', show_cmse=False)
    knn_scores.append(imp_knn_cmse)

plt.plot(k_values, knn_scores, marker='o')
plt.xlabel("K")
plt.ylabel("cMSE")
plt.title("KNN Regression: K vs. cMSE")
plt.show()

In Iterative imputation, best K for KNN regressor is 11

In [200]:
knn_reg = make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=11))
imp_iterative_model_knn, imp_iterative_cmse_knn = evaluate_imputation(X, y, c, model=knn_reg, method='iterative')

## Task 3.2 Train models that do not require imputation

### Data preprocessing, no imputation

In [37]:
df = pd.read_csv('./mlNOVA/train_data.csv')
df.drop(columns=['id'], inplace=True)
df = df.dropna(subset=['SurvivalTime'])
print(f"Remaining points: {df.shape[0]}")

In [38]:
X = df.drop(columns=['SurvivalTime', 'Censored'])
y = df['SurvivalTime']
c = df['Censored']

# Display the shapes of X and y to verify
print(f"Feature matrix X shape: {X.shape}")
print(f"Target vector y shape: {y.shape}")
print(f"Censored vector c shape: {c.shape}")

X = np.array(X)
y = np.array(y)
c = np.array(c)

# Split the data into train and test sets (with missing values)
X_train, X_test, y_train, y_test, c_train, c_test = train_test_split(X, y, c, test_size=0.2, random_state=30)

### Tree based methods

#### Function for training tree-based model

In [8]:
def train_tree_based_model(X, y, c, type='decision_tree', max_features='all', **kwargs):
    if max_features == 'all':
        max_features = X.shape[1]

    if type == 'decision_tree':
        model = DecisionTreeRegressor(random_state=30, max_features=max_features, **kwargs)
    elif type == 'random_forest':
        model = RandomForestRegressor(random_state=30, max_features=max_features, **kwargs)
    elif type == 'xgboost':
        model = XGBRegressor(random_state=30, **kwargs)
    elif type == 'catboost':
        model = CatBoostRegressor(random_state=30, **kwargs)
    elif type == 'hist_gradient_boost':
        model = HistGradientBoostingRegressor(random_state=30, **kwargs)
    else:
        raise ValueError(f"Unsupported model type: {type}")

    # Cross-validation predictions
    y_pred_cv = cross_val_predict(model, X, y, cv=10)
    
    # Calculate cMSE
    cMSE_cv = cMSE(y, y_pred_cv, c)
    print(f"{type} cMSE: {cMSE_cv}")
    
    # Train the model on the full training set
    model.fit(X, y)
    
    return model, cMSE_cv

#### Select best number of features for decision trees and random forest

In [204]:
features = ['all', 'sqrt', 'log2']
decision_tree_cMSEs = []
random_forest_cMSEs = []
hist_gradient_boost_cMSEs = []

for n_features in features:
    print(f"Number of features used per step: {n_features}")
    
    # Train Decision Tree
    dt_model, dt_cMSE = train_tree_based_model(
        X_train, y_train, c_train, type='decision_tree', max_features=n_features)
    decision_tree_cMSEs.append(dt_cMSE)
    y_dt_test = dt_model.predict(X_test)
    cMSE_dt_test = cMSE(y_test, y_dt_test, c_test)
    print(f"Decision Tree Test cMSE: {cMSE_dt_test}")

    # Train Random Forest
    rf_model, rf_cMSE = train_tree_based_model(
        X_train, y_train, c_train, type='random_forest', max_features=n_features)
    random_forest_cMSEs.append(rf_cMSE)
    y_rf_test = rf_model.predict(X_test)
    cMSE_rf_test = cMSE(y_test, y_rf_test, c_test)
    print(f"Random Forest Test cMSE: {cMSE_rf_test}")
    
    print()
    
# Plot
plt.plot(features, decision_tree_cMSEs, label="Decision Tree", marker='o')
plt.plot(features, random_forest_cMSEs, label="Random Forest", marker='o')
plt.xlabel("Num Features used per step")
plt.ylabel("Cross-Validation cMSE")
plt.title("Impact of tree type by features")
plt.legend()
plt.show()

#### Try out all tree-based models

In [52]:
depths = range(1,11)
dt_scores = []
rf_scores = []
xgb_scores = []
cat_scores = []
hgb_scores = []

for d in depths:
    # Decision Tree
    dt_model, dt_cMSE = train_tree_based_model(X_train, y_train, c_train, type='decision_tree', max_depth=d, max_features='all')
    dt_scores.append(dt_cMSE)

    # Random Forest
    rf_model, rf_cMSE = train_tree_based_model(X_train, y_train, c_train, type='random_forest', max_depth=d, max_features='sqrt')
    rf_scores.append(rf_cMSE)

    # XGBoost
    xgb_model, xgb_cMSE = train_tree_based_model(X_train, y_train, c_train, type='xgboost', max_depth=d, learning_rate=0.05)
    xgb_scores.append(xgb_cMSE)
    
    # CatBoost
    cat_model, cat_cMSE = train_tree_based_model(X_train, y_train, c_train, type='catboost', max_depth=d, learning_rate=0.01, silent=True)
    cat_scores.append(cat_cMSE)
    
    # Histogram Gradient Boosting
    hgb_model, hgb_cMSE = train_tree_based_model(X_train, y_train, c_train, type='hist_gradient_boost', max_depth=d)
    hgb_scores.append(hgb_cMSE)

In [53]:
plt.plot(depths, dt_scores, label="Decision Tree", marker='o')
plt.plot(depths, rf_scores, label="Random Forest", marker='o')
plt.plot(depths, xgb_scores, label="XGBoost", marker='o')
plt.plot(depths, cat_scores, label="CatBoost", marker='o')
plt.plot(depths, hgb_scores, label="Hist Gradient Boost", marker='o')
plt.xlabel("Max Depth Value")
plt.ylabel("Cross-Validation cMSE")
plt.title("Impact of max depth in tree-based models")
plt.legend()
plt.show()

#### Run the best models

In [ ]:
# Decision Tree
dt_model, dt_cMSE = train_tree_based_model(X_train, y_train, c_train, type='decision_tree', max_depth=2, max_features='all')
dt_scores.append(dt_cMSE)

# Random Forest
rf_model, rf_cMSE = train_tree_based_model(X_train, y_train, c_train, type='random_forest', max_depth=5, max_features='sqrt')
rf_scores.append(rf_cMSE)

# XGBoost
xgb_model, xgb_cMSE = train_tree_based_model(X_train, y_train, c_train, type='xgboost', max_depth=1, learning_rate=0.05)
dt_scores.append(xgb_cMSE)

# CatBoost
cat_model, cat_cMSE = train_tree_based_model(X_train, y_train, c_train, type='catboost', max_depth=1, learning_rate=0.01, silent=True)
cat_scores.append(cat_cMSE)

# Histogram Gradient Boosting
hgb_model, hgb_cMSE = train_tree_based_model(X_train, y_train, c_train, type='hist_gradient_boost', max_depth=1)
hgb_scores.append(hgb_cMSE)

#### Compare CMSE across models

In [30]:
models = ['Decision Tree', 'Random Forest', 'XGBoost', 'CatBoost', 'HistGradientBoost']
cMSE_values = [dt_cMSE, rf_cMSE, xgb_cMSE, cat_cMSE, hgb_cMSE]

plt.bar(models, cMSE_values)
plt.xlabel("Model")
plt.ylabel("Cross-Validation cMSE")
plt.title("Comparison of Tree-Based Models")
plt.xticks(rotation=45)
plt.show()

### CatBoost: Accelerated Failure Time (AFT) CatBoost applied to the assignment data.

#### Pre-processing

In [12]:
df = pd.read_csv('./mlNOVA/train_data.csv')
df.drop(columns=['id'], inplace=True)
df.head()

In [13]:
df = df.dropna(subset=['SurvivalTime'])

X = df.drop(columns=['SurvivalTime', 'Censored'])
y = df['SurvivalTime']
c = df['Censored']

# Display the shapes of X and y to verify
print(f"Feature matrix X shape: {X.shape}")
print(f"Target vector y shape: {y.shape}")
print(f"Censored vector c shape: {c.shape}")

X = np.array(X)
y = np.array(y)
c = np.array(c)
# Split the data into train and test sets (with missing values)
X_train, X_test, y_train, y_test, c_train, c_test = train_test_split(X, y, c, test_size=0.2, random_state=42)

df['y_lower'] = df['SurvivalTime']
df['y_upper'] = np.where(df['Censored'], df['SurvivalTime'], -1)
stratifying_column = df['Censored']
df.drop(columns=['Censored','SurvivalTime'], inplace=True)
# Replace NaNs with -1 to indicate a distinct "missing" category
df['TreatmentResponse'] = df['TreatmentResponse'].fillna(-1).astype(int)
df.head()

In [14]:
train, test = train_test_split(df, test_size=0.2, stratify=stratifying_column, random_state=30)

features = df.columns.difference(['y_lower', 'y_upper'], sort=False)
cat_features = ['Gender', 'Stage', 'TreatmentType', 'TreatmentResponse']

train_pool = Pool(train[features], label=train.loc[:,['y_lower','y_upper']], cat_features=cat_features)
test_pool = Pool(test[features], label=test.loc[:,['y_lower','y_upper']], cat_features=cat_features)

#### Models

In [15]:
model_normal = CatBoostRegressor(iterations=500,
                                 loss_function='SurvivalAft:dist=Normal',
                                 eval_metric='SurvivalAft',
                                 verbose=0
                                )

model_logistic = CatBoostRegressor(iterations=500,
                                 loss_function='SurvivalAft:dist=Logistic;scale=1.2',
                                 eval_metric='SurvivalAft',
                                 verbose=0)

model_extreme = CatBoostRegressor(iterations=500,
                                 loss_function='SurvivalAft:dist=Extreme;scale=2',
                                 eval_metric='SurvivalAft',
                                 verbose=0)

In [16]:
model_normal.fit(train_pool, eval_set=test_pool)
model_logistic.fit(train_pool, eval_set=test_pool)
model_extreme.fit(train_pool, eval_set=test_pool)

In [242]:
train_predictions = pd.DataFrame({'y_lower': train['y_lower'],
                                  'y_upper': train['y_upper'],
                                  'preds_normal': model_normal.predict(train_pool, prediction_type='Exponent'),
                                  'preds_logistic': model_logistic.predict(train_pool, prediction_type='Exponent'),
                                  'preds_extreme': model_extreme.predict(train_pool, prediction_type='Exponent')})
train_predictions['y_upper'] = np.where(train_predictions['y_upper']==-1, np.inf, train_predictions['y_upper'])

test_predictions = pd.DataFrame({'y_lower': test['y_lower'],
                                  'y_upper': test['y_upper'],
                                  'preds_normal': model_normal.predict(test_pool, prediction_type='Exponent'),
                                  'preds_logistic': model_logistic.predict(test_pool, prediction_type='Exponent'),
                                  'preds_extreme': model_extreme.predict(test_pool, prediction_type='Exponent')})
test_predictions['y_upper'] = np.where(test_predictions['y_upper']==-1, np.inf, test_predictions['y_upper'])

In [140]:
def interval_mae(y_true_lower, y_true_upper, y_pred):
    mae = np.where((y_true_lower <= y_pred) & (y_pred <= y_true_upper),
                   0,
                   np.minimum(np.abs(y_true_lower-y_pred),
                              np.abs(y_true_upper-y_pred))) 
    return mae.mean()

In [141]:
distributions = ['normal', 'logistic', 'extreme']
print('Interval MAE')
for dist in distributions:
    train_metric = interval_mae(train_predictions['y_lower'], train_predictions['y_upper'], train_predictions[f'preds_{dist}'])
    test_metric = interval_mae(test_predictions['y_lower'], test_predictions['y_upper'], test_predictions[f'preds_{dist}'])
    print(f'Train set. dist:{dist}: {train_metric:0.2f}')
    print(f'Test set. dist:{dist}: {test_metric:0.2f}')
    print('---------------------------')

#### Test cMSE for all models

In [17]:
y_normal_pred_test = model_normal.predict(test_pool, prediction_type='Exponent')
cMSE_normal_test = cMSE(y_test, y_normal_pred_test, c_test)
print(f"Test Set Censored Mean Squared Error (cMSE): {cMSE_normal_test}")

In [18]:
y_logistic_pred_test = model_logistic.predict(test_pool, prediction_type='Exponent')
cMSE_logistic_test = cMSE(y_test, y_logistic_pred_test, c_test)
print(f"Test Set Censored Mean Squared Error (cMSE): {cMSE_logistic_test}")

In [19]:
y_extreme_pred_test = model_extreme.predict(test_pool, prediction_type='Exponent')
cMSE_extreme_test = cMSE(y_test, y_extreme_pred_test, c_test)
print(f"Test Set Censored Mean Squared Error (cMSE): {cMSE_extreme_test}")

## Task 3.3 Evaluation

### Pre-processing

In [298]:
df = pd.read_csv('./mlNOVA/train_data.csv')
df.drop(columns=['id'], inplace=True)
df = df.dropna(subset=['SurvivalTime'])
print(f"Remaining points after removing unlabeled rows: {df.shape[0]}")

In [299]:
X = df.drop(columns=['SurvivalTime', 'Censored'])
y = df['SurvivalTime']
c = df['Censored']

# Display the shapes of X and y to verify
print(f"Feature matrix X shape: {X.shape}")
print(f"Target vector y shape: {y.shape}")
print(f"Censored vector c shape: {c.shape}")

X = np.array(X)
y = np.array(y)
c = np.array(c)

### Imput and run models

#### Simple imputer with mean strategy

##### Linear regression

In [214]:
linear_reg = make_pipeline(StandardScaler(), LinearRegression())
lr_model, lr_cMSE = evaluate_imputation(X, y, c, model=linear_reg, method='simple', strategy='mean', show_metrics=False, show_plot=False)

##### Decision tree

In [300]:
dt_model, dt_cMSE = evaluate_imputation(X, y, c, model=dt_model, method='simple', strategy='mean', show_metrics=False, show_plot=False)

##### Random forest

In [301]:
rf_model, rf_cMSE = evaluate_imputation(X, y, c, model=rf_model, method='simple', strategy='mean', show_metrics=False, show_plot=False)

##### XGBoost

In [302]:
xgb_model, xgb_cMSE = evaluate_imputation(X, y, c, model=xgb_model, method='simple', strategy='mean', show_metrics=False, show_plot=False)

##### CatBoost

In [311]:
cat_model, cat_cMSE = evaluate_imputation(X, y, c, model=cat_model, method='simple', strategy='mean', show_metrics=False, show_plot=False)

##### Histogram Gradient Boost

In [312]:
hgb_model, hgb_cMSE = evaluate_imputation(X, y, c, model=hgb_model, method='simple', strategy='mean', show_metrics=False, show_plot=False)

##### Compare models through CV cMSE

In [313]:
models = ['Linear Regression', 'Decision Tree', 'Random Forest', 'XGBoost', 'CatBoost', 'HistGradientBoost']
cMSE_values = [lr_cMSE, dt_cMSE, rf_cMSE, xgb_cMSE, cat_cMSE, hgb_cMSE]

plt.bar(models, cMSE_values)
plt.xlabel("Model")
plt.ylabel("Cross-Validation cMSE")
plt.title("Comparison of Models")
plt.xticks(rotation=45)
plt.show()

#### KNN imputer with K=19

##### Linear Regression

In [306]:
linear_reg = make_pipeline(StandardScaler(), LinearRegression())
lr_model, lr_cMSE = evaluate_imputation(X, y, c, model=linear_reg, method='knn', n_neighbors=19)

##### Decision Tree

In [328]:
dt_model, dt_cMSE = evaluate_imputation(X, y, c, model=dt_model, method='knn', n_neighbors=19)

##### Random forest

In [315]:
rf_model, rf_cMSE = evaluate_imputation(X, y, c, model=rf_model, method='knn', n_neighbors=19)

##### XGBoost

In [329]:
xgb_model, xgb_cMSE = evaluate_imputation(X, y, c, model=xgb_model, method='knn', n_neighbors=19)

##### CatBoost

In [330]:
cat_model, cat_cMSE = evaluate_imputation(X, y, c, model=cat_model, method='knn', n_neighbors=19)

##### Histogram Gradient Boost

In [318]:
hgb_model, hgb_cMSE = evaluate_imputation(X, y, c, model=hgb_model, method='knn', n_neighbors=19)

##### Compare models through CV cMSE

In [319]:
models = ['Linear Regression', 'Decision Tree', 'Random Forest', 'XGBoost', 'CatBoost', 'HistGradientBoost']
cMSE_values = [lr_cMSE, dt_cMSE, rf_cMSE, xgb_cMSE, cat_cMSE, hgb_cMSE]

plt.bar(models, cMSE_values)
plt.xlabel("Model")
plt.ylabel("Cross-Validation cMSE")
plt.title("Comparison of Models")
plt.xticks(rotation=45)
plt.show()

#### Iterative Imputer

##### Linear Regression

In [320]:
linear_reg = make_pipeline(StandardScaler(), LinearRegression())
lr_model, lr_cMSE = evaluate_imputation(X, y, c, model=linear_reg, method='iterative')

##### Decision Tree

In [321]:
dt_model, dt_cMSE = evaluate_imputation(X, y, c, model=dt_model, method='iterative')

##### Random Forest

In [322]:
rf_model, rf_cMSE = evaluate_imputation(X, y, c, model=rf_model, method='iterative')

##### XGBoost

In [324]:
xgb_model, xgb_cMSE = evaluate_imputation(X, y, c, model=xgb_model, method='iterative')

##### CatBoost

In [325]:
cat_model, cat_cMSE = evaluate_imputation(X, y, c, model=cat_model, method='iterative')

##### Histogram Gradient Boost

In [326]:
hgb_model, hgb_cMSE = evaluate_imputation(X, y, c, model=hgb_model, method='iterative')

##### Compare models through CV cMSE

In [327]:
models = ['Linear Regression', 'Decision Tree', 'Random Forest', 'XGBoost', 'CatBoost', 'HistGradientBoost']
cMSE_values = [lr_cMSE, dt_cMSE, rf_cMSE, xgb_cMSE, cat_cMSE, hgb_cMSE]

plt.bar(models, cMSE_values)
plt.xlabel("Model")
plt.ylabel("Cross-Validation cMSE")
plt.title("Comparison of Models")
plt.xticks(rotation=45)
plt.show()

### Submission of best models

#### Pre-processing

In [32]:
df = pd.read_csv('./mlNOVA/train_data.csv')
df.drop(columns=['id'], inplace=True)
# drop unlabeled rows (missing Survival Time)
df = df.dropna(subset=['SurvivalTime'])

X = df.drop(columns=['SurvivalTime', 'Censored'])
y = df['SurvivalTime']
c = df['Censored']

# Display the shapes of X and y to verify
print(f"Feature matrix X shape: {X.shape}")
print(f"Target vector y shape: {y.shape}")
print(f"Censored vector c shape: {c.shape}")

X = np.array(X)
y = np.array(y)
c = np.array(c)

#### Models

KNN Imputer with K=19

##### Decision tree

In [335]:
dt_model_knn, dt_cMSE_knn = evaluate_imputation(X, y, c, model=dt_model, method='knn', n_neighbors=19)

##### Random Forest

In [47]:
rf_model_knn, rf_cMSE_knn = evaluate_imputation(X, y, c, model=rf_model, method='knn', n_neighbors=19)

##### XGBoost

In [57]:
xgb_model_knn, xgb_cMSE_knn = evaluate_imputation(X, y, c, model=xgb_model, method='knn', n_neighbors=19)

##### CatBoost

In [ ]:
cat_model, cat_cMSE = train_tree_based_model(X_train, y_train, c_train, type='catboost', max_depth=1, learning_rate=0.01, silent=True)

cat_model_knn, cat_cMSE_knn = evaluate_imputation(X, y, c, model=cat_model, method='knn', n_neighbors=19)

##### Histogram Gradient Boost

In [55]:
hgb_model_knn, hgb_cMSE_knn = evaluate_imputation(X, y, c, model=hgb_model, method='knn', n_neighbors=19)

CV cMSE: 2.668070624249363
Test cMSE: 2.5242629158668666


#### Load Kaggle dataset and impute it

In [48]:
X_kaggle_df = pd.read_csv('./mlNOVA/test_data.csv')
X_kaggle_df.drop(columns=['id'], inplace=True)
X_kaggle_df.head()

,Age,Gender,Stage,GeneticRisk,TreatmentType,ComorbidityIndex,TreatmentResponse
0,71.0,1,4,NaN,1,1.0,1.0
1,57.0,1,4,0.0,0,NaN,0.0
2,66.0,0,2,0.0,0,0.0,1.0
3,73.0,0,3,NaN,1,1.0,1.0
4,53.0,1,4,0.0,0,0.0,0.0


Impute dataset

In [49]:
imputer_sm = SimpleImputer(strategy='mean')
imputer_knn = KNNImputer(n_neighbors=19)
imputer_iter = IterativeImputer()

X_imp_sm = imputer_sm.fit_transform(X_kaggle_df)
X_imp_knn = imputer_knn.fit_transform(X_kaggle_df)
X_imp_iter = imputer_iter.fit_transform(X_kaggle_df)

#### Make prediction

In [83]:
y_cat_knn_pred = cat_model_knn.predict(X_imp_knn)
save_predictions(y_cat_knn_pred, './submissions/handle-missing-submission-10.csv')